# Fault Tolerance: RDD Lineage and Partition Recomputation

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 01_lineage_and_recomputation.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark recovers lost partitions via deterministic RDD lineage graph re-evaluation without data replication.
- **Why does this topic exist?**: Lineage recomputation is Spark's core resilience mechanism.
- **What problem does it solve?**: Understanding what gets recomputed when worker nodes fail.
- **Where is it used in real-world Data Engineering?**: Spot instances and preemptible cloud clusters.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | RDD dependencies, shuffle, stages |
| **Required depth** | Working knowledge: can explain how Spark recovers a lost partition |
| **Data Engineer** | Explain lineage-based recovery and its cost. |
| **Senior Data Engineer** | Design deterministic pipelines and place checkpoints where recomputation is expensive. |
| **Spark specialist** | Reason about narrow/wide dependencies and shuffle output loss. |
| **Next topic** | `11_fault_tolerance_and_reliability/02_executor_failure.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Spark does not replicate every intermediate dataset. Instead, it records how each partition was derived and
recomputes lost partitions from the lineage. That is cheap for narrow pipelines and expensive after a shuffle,
where losing map outputs can force a large parent stage to run again. Lineage is the reason Spark can be resilient
without copying everything, and the reason non-deterministic transformations are dangerous.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Lineage** | Recipe for rebuilding a partition | Parent RDDs and the transformation applied |
| **Narrow dependency** | One parent partition | Recompute only the affected child partition |
| **Wide dependency** | Many parent partitions | May require a shuffle and a larger recomputation |
| **Recomputation** | Re-execute lost work | Driven by the scheduler after a failure |
| **Determinism** | Same input, same output | Required for correct recomputation |
| **Checkpoint** | Cut the lineage | Persist a stable intermediate result |

### How it works, step by step

1. A task fails or an executor is lost.
2. The scheduler identifies the lost partitions.
3. It walks the lineage to find the nearest available parent data.
4. It resubmits tasks to recompute the missing partitions.
5. If shuffle map outputs were lost, the parent stage may be recomputed.
6. The job continues once the missing data is rebuilt.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Lineage Recomputation Mechanics: If partition P2 fails, Spark traces P2's parent lineage back to source files and recomputes only P2.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Lineage_Recompute_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rdd = spark.sparkContext.parallelize(range(50), 4).map(lambda x: x*2).filter(lambda x: x > 20)
print("Lineage graph of transformations:")
print(rdd.toDebugString().decode('utf-8'))
print(f"Computed count: {rdd.count()}")

---

## Experiment: Lineage Recomputation Verification

- **1. What are we testing?**: Verify that lost partitions are reconstructed from upstream lineage.
- **2. What do we expect?**: Recomputation succeeds deterministically.

In [ ]:
print('Lineage graph has depth:', len(rdd.toDebugString().splitlines()))

**3. What happened?**: Lineage tree verified.  
**4. Why did it happen?**: Spark retains the complete sequence of deterministic operations.  
**5. Practical Engineering Lesson**: Deterministic transformations ensure exact partition reproduction upon failure.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Risk model recomputation

- **Industry**: Financial services
- **Scenario**: A long feature pipeline fails near the end after a node restart.
- **Problem**: Recomputing the entire lineage is expensive and delays the risk run.
- **Why this concept is relevant**: Checkpointing a stable feature stage bounds the recomputation to the steps after the checkpoint.
- **Implementation idea**: Write the feature stage to reliable storage, read it back, and continue; compare recovery time.
- **Expected result**: Faster recovery with a known materialization cost.
- **Alternative**: Persist the feature stage as a table.
- **Trade-offs**: A table is durable and shareable; a checkpoint is tied to the job.

### Use Case 2: Non-deterministic pipeline bug

- **Industry**: Advertising
- **Scenario**: A job produces different results after a retry.
- **Problem**: The pipeline uses a random value or current timestamp inside a transformation, so recomputation changes the output.
- **Why this concept is relevant**: Lineage recovery assumes deterministic transformations; non-determinism breaks that assumption.
- **Implementation idea**: Seed randomness, capture the timestamp once, and test retry behavior.
- **Expected result**: Reproducible results across failures.
- **Alternative**: Materialize the random assignment once.
- **Trade-offs**: Materialization makes the assignment stable but adds a storage step.

---

## Business Value

- Recovery without replicating every intermediate dataset.
- Clear cost model for failures.
- Better checkpoint placement.
- Awareness of determinism risks.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Keep transformations deterministic.
- Checkpoint long lineages and iterative jobs.
- Expect larger recomputation after shuffle-heavy stages.
- Monitor stage retries and lost shuffle outputs.
- Do not rely on lineage for cross-job recovery.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Observe partition recomputation

**Steps**

1. Run a multi-stage job and note the stage IDs.
2. Simulate a task failure or stop an executor during execution.
3. Watch which stages are resubmitted in the UI.

**What to observe**: Only the lost partitions and their dependencies are recomputed; shuffle loss can trigger a parent stage.

**Healthy result**: The job completes with a bounded recomputation.

**Problematic result**: A small failure recomputes most of the pipeline.

**Engineering takeaway**: Checkpoint before expensive shuffle boundaries when recovery cost matters.

---

## Trade-offs and Alternatives

### Advantages

- No full replication
- Automatic recovery
- Fine-grained recomputation

### Limitations

- Recomputation can be expensive
- Non-determinism breaks correctness
- Shuffle loss is costly

### When to use it

- Transient executor failures
- Long ETL pipelines
- Iterative jobs

### When not to use it

- Random/timestamp logic in transformations
- Assuming lineage survives job failure
- Ignoring retry cost

### Alternatives inside Spark

- Lineage
- DAGScheduler
- Checkpointing

### Alternatives outside Spark

- Replicated storage
- Workflow retries
- Idempotent sinks

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Lineage Depth
**Problem Statement**: Count transformation steps in toDebugString.  
**Input / Context**: rdd

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Non-Deterministic Hazard
**Problem Statement**: Why random numbers break lineage recomputation.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Lineage Truncation
**Problem Statement**: When to truncate lineage.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### Results changed after a retry

**Scenario**: A job's output differs between the original run and a retry after a failure.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Look for random values, current timestamps, or non-deterministic UDFs.
2. Check whether the transformation is evaluated more than once.
3. Seed randomness or materialize the non-deterministic result once.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Lineage Depth):
```python
print(len(rdd.toDebugString().splitlines()))
```

#### Solution 2 (Non-Deterministic Hazard):
```python
print('Recomputed partitions will have different random values, corrupting joins.')
```

#### Solution 3 (Lineage Truncation):
```python
print('Use checkpoint() when lineage depth exceeds ~50 steps to prevent StackOverflowError.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Lineage Resilience**: Recomputes lost partitions on the fly.
- **Zero Replication**: Saves network bandwidth compared to HDFS replication.

### Essential Technical Terms
- `Lineage`: Acyclic transformation graph.
- `Recomputation`: Evaluating parent steps for lost slices.

### Common Engineering Mistakes
- **Non-deterministic functions**: e.g. random.random() inside lineage causes corrupt recomputation.

### Interview Questions on This Topic
- **Q: How does Spark handle a lost partition?**
  - *A*: It recomputes only the lost partition from parent dependencies using its lineage graph.


### More Interview Questions

- **Q: How does Spark recover a lost partition?**
  - *A*: It uses the RDD/DataFrame lineage to recompute the missing partition from its parents, rather than restoring a replicated copy.
- **Q: Why is non-determinism dangerous for lineage recovery?**
  - *A*: If a transformation produces different values on recomputation, the recovered partition may not match the original, breaking correctness.

### What to Study Next
Open 02_executor_failure.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')